In [1]:
# takes in data from process_inputs

In [1]:
# Standard library
import warnings
import ast
from pathlib import Path
from datetime import timedelta
from abc import ABC, abstractmethod

# Third-party libraries
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind, skew
from joblib import Parallel, delayed
import xgboost as xgb

# scikit-learn
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    RepeatedStratifiedKFold,
    GridSearchCV,
    cross_val_score
)


from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    make_scorer
)

# Custom modules
from cashflow import ScorableModelTemplate

# Suppress warnings
warnings.filterwarnings("ignore")

def process_inputs(raw_consumer_file: str, raw_transactions_file: str):
    """Input argument will vary. See you competition's template.

    :param raw_files: list of file path strings, depends on competition
    :return: anything needed for you model to make predictions, e.g. features or processed data
    """
    # Read in files
    df_consumer = pd.read_parquet(raw_consumer_file)
    df_transactions = pd.read_parquet(raw_transactions_file)

    # Merge and filter to get only valid data
    merged_df = df_transactions.merge(df_consumer, on = "masked_consumer_id")
    filtered_df = merged_df[merged_df["posted_date"]< merged_df["evaluation_date"]]

    # Clean evaluation_date and create evaluation_day for day of week of evaluation_date
    filtered_df['evaluation_date'] = pd.to_datetime(filtered_df['evaluation_date'])
    filtered_df['evaluation_day'] = filtered_df['evaluation_date'].dt.dayofweek



    # Clean posted_date and create posted_day for day of week of posted_date
    filtered_df['posted_date'] = pd.to_datetime(filtered_df['posted_date'])
    filtered_df['posted_day'] = filtered_df['posted_date'].dt.dayofweek

    # Get loan category from masked_consumer_id
    filtered_df['loan_category'] = filtered_df['masked_consumer_id'].str[2].astype(int)
    filtered_df['converted_date'] = (max(filtered_df['posted_date']) - filtered_df['posted_date']).dt.total_seconds()/3600
    
    return filtered_df


def compute_trend(series):
    if not isinstance(series, list):
        return 0  # fallback if data malformed

    series = np.array(series)
    nonzero_idx = np.nonzero(series)[0]
    nonzero_vals = series[nonzero_idx]

    if len(nonzero_vals) <= 2:
        return 0

    X = nonzero_idx.reshape(-1, 1)  # time indices
    y = nonzero_vals
    model = LinearRegression().fit(X, y)
    return model.coef_[0]  # slope

def process_user_parallel(user_id, group, ranges=24*7, num_categories=36):
    start = group['converted_date'].min()
    end = group['converted_date'].max()
    
    segments = []
    for x in range(int(start), int(end), ranges):
         mask = group[(group['converted_date'] >= x) & (group['converted_date'] < x + ranges)]
         if len(mask) == 0 or (mask['converted_date'].min() + ranges >  end):
             continue

         data = {
             'net_week_series': mask['amount'].sum(),
             'count_week_series': mask['amount'].count(),
             'inflow_week_series': mask[mask['amount'] > 0]['amount'].sum(),
             'outflow_week_series': mask[mask['amount'] < 0]['amount'].sum(),
         }
         for i in range(num_categories):
             data[f"week_series_category_{i}"] = mask[mask['category'] == i]['amount'].sum()
         segments.append(data)

    if len(segments) == 0:
        return user_id, pd.Series({
            key: [0] 
            for key in ['net_week_series', 'count_week_series', 'inflow_week_series', 'outflow_week_series'] +
                        [f"week_series_category_{i}" for i in range(num_categories)]
        })

    segments = segments[::-1]
    output = {key: [seg[key] for seg in segments] for key in segments[0].keys()}
    return user_id, pd.Series(output)

df_train = pd.read_csv("/home/jkaminsky/private/DSC 190/mlc/pre_saved_data/df_train.csv")

#create transaction_grouped
grouped = list(df_train.groupby('masked_consumer_id'))
results = Parallel(n_jobs=-1)(delayed(process_user_parallel)(uid, grp) for uid, grp in grouped)
transaction_grouped = pd.DataFrame({uid: out for uid, out in results}).T.reset_index().rename(columns={'index': 'masked_consumer_id'})
for col in transaction_grouped.columns[1:]:
    transaction_grouped[col] = transaction_grouped[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# Apply to each time series column
trend_df = pd.DataFrame()
trend_df['masked_consumer_id'] = transaction_grouped['masked_consumer_id']

# All time-series columns
series_cols = transaction_grouped.columns.drop('masked_consumer_id')

for col in series_cols:
    trend_df[col + '_trend'] = transaction_grouped[col].apply(compute_trend)

In [2]:
df_train = df_train.merge(trend_df, on = 'masked_consumer_id', how = 'left')

build_df = df_train.groupby('masked_consumer_id').agg({"total_balance": "mean", "week_series_category_0_trend":"mean", 
                                                       "week_series_category_1_trend":"mean", "week_series_category_2_trend":"mean",
                                                       "week_series_category_3_trend":"mean", "week_series_category_4_trend":"mean",
                                                       "week_series_category_5_trend":"mean", "week_series_category_6_trend":"mean",
                                                       "week_series_category_7_trend":"mean", "week_series_category_8_trend":"mean",
                                                       "week_series_category_9_trend":"mean", "week_series_category_10_trend":"mean",
                                                       "week_series_category_11_trend":"mean", "week_series_category_12_trend":"mean",
                                                       "week_series_category_13_trend":"mean", "week_series_category_14_trend":"mean",
                                                       "week_series_category_15_trend":"mean", "week_series_category_16_trend":"mean",
                                                       "week_series_category_17_trend":"mean", "week_series_category_18_trend":"mean",
                                                       "week_series_category_19_trend":"mean", "week_series_category_20_trend":"mean",
                                                       "week_series_category_21_trend":"mean", "week_series_category_22_trend":"mean",
                                                       "week_series_category_23_trend": "mean", "week_series_category_24_trend":"mean",
                                                       "week_series_category_25_trend": "mean", "week_series_category_26_trend":"mean",
                                                       "week_series_category_27_trend": "mean", "week_series_category_28_trend":"mean",
                                                       "week_series_category_29_trend": "mean", "week_series_category_30_trend":"mean",
                                                       "week_series_category_31_trend": "mean", "week_series_category_32_trend":"mean",
                                                       "week_series_category_33_trend": "mean", "week_series_category_34_trend": "mean",
                                                       "week_series_category_35_trend": "mean",
                                                       "FPF_TARGET":"first"}).reset_index()



In [3]:
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, kurtosis, skew
from collections import Counter
from sklearn.linear_model import LinearRegression

def gini(array):
    array = np.sort(np.abs(array))
    index = np.arange(1, array.shape[0] + 1)
    n = array.shape[0]
    return (np.sum((2 * index - n - 1) * array)) / (n * np.sum(array) + 1e-9)

def compute_trend(x, y):
    """
    x: datetime-like index (e.g., DatetimeIndex)
    y: numeric values (e.g., average spend over time)
    """
    if len(x) < 2:
        return 0
    # Convert datetime index to days since min date
    x_days = (x - x.min()).days  # x must be a DatetimeIndex
    x_arr = np.array(x_days).reshape(-1, 1)
    y_arr = np.array(y)
    try:
        model = LinearRegression().fit(x_arr, y_arr)
        return model.coef_[0]
    except:
        return 0

def compute_user_statistics(df_txn, suffix):
    df_txn['posted_date'] = pd.to_datetime(df_txn['posted_date'])
    df_txn['log_amount'] = np.log1p(np.abs(df_txn['amount']))
    df_txn['is_positive'] = df_txn['amount'] > 0
    df_txn['year_month'] = df_txn['posted_date'].dt.to_period('M')
    df_txn['week'] = df_txn['posted_date'].dt.isocalendar().week
    df_txn['dayofweek'] = df_txn['posted_date'].dt.dayofweek

    features = []

    for user_id, group in df_txn.groupby("masked_consumer_id"):
        amounts = group["amount"]
        dates = group["posted_date"].sort_values()
        gaps = dates.diff().dt.total_seconds().dropna() / (60*60*24)
        cat_counts = group['category'].value_counts(normalize=True)

        # Time grouping
        weekly = group.groupby(group['posted_date'].dt.to_period('W'))['amount']
        monthly = group.groupby('year_month')['amount']
        weekday = group.groupby('dayofweek')['amount']

        features.append({
            f'masked_consumer_id': user_id,
            f'amount_mean{suffix}': amounts.mean(),
            f'amount_median{suffix}': amounts.median(),
            f'amount_std{suffix}': amounts.std(),
            f'amount_var{suffix}': amounts.var(),
            f'amount_min{suffix}': amounts.min(),
            f'amount_max{suffix}': amounts.max(),
            f'amount_skew{suffix}': skew(amounts),
            f'amount_kurtosis{suffix}': kurtosis(amounts),
            f'amount_cv{suffix}': amounts.std() / (amounts.mean() + 1e-6),
            f'amount_gini{suffix}': gini(amounts.values),
            f'log_amount_mean{suffix}': group['log_amount'].mean(),
            f'txn_count{suffix}': len(group),
            f'txn_positive_ratio{suffix}': group['is_positive'].mean(),
            f'txn_duration_days{suffix}': (dates.max() - dates.min()).days + 1,
            f'txn_active_days{suffix}': dates.nunique(),
            f'txn_active_day_pct{suffix}': dates.nunique() / ((dates.max() - dates.min()).days + 1 + 1e-6),
            f'txn_gap_iqr{suffix}': np.subtract(*np.percentile(gaps, [75, 25])) if len(gaps) > 1 else 0,
            f'txn_burstiness{suffix}': gaps.std() if len(gaps) > 1 else 0,
            f'cat_count{suffix}': group['category'].nunique(),
            f'cat_concentration{suffix}': cat_counts.iloc[0] if len(cat_counts) > 0 else 0,
            f'net_spending_ratio{suffix}': amounts[amounts > 0].sum() / (amounts.abs().sum() + 1e-6),
            f'high_to_low_spend_ratio{suffix}': amounts.quantile(0.9) / (amounts.quantile(0.1) + 1e-6) if amounts.quantile(0.1) != 0 else 0,
            f'outlier_count{suffix}': ((amounts - amounts.mean()).abs() > 3 * amounts.std()).sum(),
            f'low_dollar_txn_pct{suffix}': (amounts.abs() < 5).mean(),

            # Temporal Features
            f'weekly_avg_spend{suffix}': weekly.mean().mean(),
            f'weekly_txn_freq{suffix}': weekly.count().mean(),
            f'weekly_spend_slope{suffix}': compute_trend(weekly.mean().index.to_timestamp(), weekly.mean()),
            f'monthly_avg_spend{suffix}': monthly.mean().mean(),
            f'monthly_txn_freq{suffix}': monthly.count().mean(),
            f'monthly_spend_slope{suffix}': compute_trend(monthly.mean().index.to_timestamp(), monthly.mean()),
            f'weekday_spend_std{suffix}': weekday.mean().std(),
            f'weekend_spend_pct{suffix}': group[group['dayofweek'] >= 5]['amount'].sum() / (amounts.sum() + 1e-6),
        })

    df_features = pd.DataFrame(features)
    return df_features
    df = df_features.merge(df_txn[['masked_consumer_id', "FPF_TARGET"]].drop_duplicates(), on='masked_consumer_id', how='left')

    results = []
    feature_cols = [col for col in df.columns if col not in ['masked_consumer_id', 'FPF_TARGET']]

    for col in feature_cols:
        group0 = df[df['FPF_TARGET'] == 0][col].dropna()
        group1 = df[df['FPF_TARGET'] == 1][col].dropna()
        if len(group0) > 0 and len(group1) > 0:
            try:
                stat, p = mannwhitneyu(group0, group1, alternative='two-sided')
                results.append({'feature': col, 'p_value': p, 'mean_0': group0.mean(), 'mean_1': group1.mean()})
            except:
                continue

    results_df = pd.DataFrame(results).sort_values('p_value')
    return df_features, results_df


In [4]:
build_df= build_df.merge(compute_user_statistics(df_train, 'ovr'), on = 'masked_consumer_id', how = 'left')
build_df= build_df.merge(compute_user_statistics(df_train[df_train['amount'] <= 0], '_ovr_neg'), on = 'masked_consumer_id', how = 'left')
build_df= build_df.merge(compute_user_statistics(df_train[df_train['amount'] > 0], '_ovr_pos'), on = 'masked_consumer_id', how = 'left')

have_31 = set(list(df_train[df_train['category'] == 31]['masked_consumer_id']))

build_df['cat_31'] = np.where(build_df['masked_consumer_id'].isin(have_31), 1, 0)

## CONDUCT BY CATEGORY, CATEGORY GROUP
for i in range(36):
    if i == 31:
        continue
    one = compute_user_statistics(df_train[df_train['category'] == i], f'_{i}')
    if len(one) > 0:
        build_df= build_df.merge(one, on = 'masked_consumer_id', how = 'left')
    two = compute_user_statistics(df_train[(df_train['category'] == i) & (df_train['amount'] <= 0)], f'_{i}_neg')
    if len(two) > 0:
        build_df= build_df.merge(two, on = 'masked_consumer_id', how = 'left')
    three = compute_user_statistics(df_train[(df_train['category'] == i) & (df_train['amount'] > 0)], f'_{i}_pos')
    if len(three) > 0:
        build_df= build_df.merge(three, on = 'masked_consumer_id', how = 'left')

    print(i)

ct = 0
for group in [[32, 34, 24, 31, 13],[14, 18],[27, 13],[17, 36],[21, 20],[33, 29, 28], [24, 32, 34, 36, 29, 21, 27, 22, 31, 26],[14, 18, 28, 20, 30, 16, 19, 23, 25, 15],[25, 23, 10, 35, 36, 26, 12],[3, 5, 2, 6, 9, 8, 7, 11, 13, 0, 1]]:
    ct += 1
    one = compute_user_statistics(df_train[df_train['category'].isin(group)], f'_group_{ct}')
    if len(one) > 0:
        build_df= build_df.merge(one, on = 'masked_consumer_id', how = 'left')
    two = compute_user_statistics(df_train[(df_train['category'].isin(group)) & (df_train['amount'] <= 0)], f'_group_{ct}_neg')
    if len(two) > 0:
        build_df= build_df.merge(two, on = 'masked_consumer_id', how = 'left')
    three = compute_user_statistics(df_train[(df_train['category'].isin(group)) & (df_train['amount'] > 0)], f'_group_{ct}_pos')
    if len(three) > 0:
        build_df= build_df.merge(three, on = 'masked_consumer_id', how = 'left')
    print(ct)




0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
32
33
34
35
1
2
3
4
5
6
7
8
9
10


In [11]:
#percentage by group (abs, total, pos only, neg only), ranking categories
from scipy.stats import mannwhitneyu

def compute_category_spending_distribution(df_txn):
    """
    Compute category-level spending distributions and test for significant differences
    between target groups (FPF_TARGET = 0 and 1).
    
    Returns:
        df_user_cat_stats: DataFrame with one row per user and percentage/rank features
        results_df: Mann-Whitney U test results with p-values for significance
    """
    df_txn['posted_date'] = pd.to_datetime(df_txn['posted_date'])
    df_txn['amount_abs'] = df_txn['amount'].abs()
    df_txn['amount_pos'] = df_txn['amount'].clip(lower=0)
    df_txn['amount_neg'] = df_txn['amount'].clip(upper=0).abs()

    user_cat_group = df_txn.groupby(['masked_consumer_id', 'category']).agg(
        amount_total=('amount', 'sum'),
        amount_abs=('amount_abs', 'sum'),
        amount_pos=('amount_pos', 'sum'),
        amount_neg=('amount_neg', 'sum'),
    ).reset_index()

    # Calculate total per user for normalizing
    user_totals = user_cat_group.groupby('masked_consumer_id').agg(
        total_amount_total=('amount_total', 'sum'),
        total_amount_abs=('amount_abs', 'sum'),
        total_amount_pos=('amount_pos', 'sum'),
        total_amount_neg=('amount_neg', 'sum'),
    ).reset_index()

    # Merge totals back into category group
    user_cat_group = user_cat_group.merge(user_totals, on='masked_consumer_id', how='left')

    # Compute percentage spend per category
    for col in ['amount_total', 'amount_abs', 'amount_pos', 'amount_neg']:
        user_cat_group[f'pct_{col}'] = user_cat_group[col] / (user_cat_group[f'total_{col}'] + 1e-6)

    # Compute per-user category rankings (lower rank = higher spend)
    user_cat_group['rank_total'] = user_cat_group.groupby('masked_consumer_id')['amount_total'].rank(ascending=False)
    user_cat_group['rank_abs'] = user_cat_group.groupby('masked_consumer_id')['amount_abs'].rank(ascending=False)
    user_cat_group['rank_pos'] = user_cat_group.groupby('masked_consumer_id')['amount_pos'].rank(ascending=False)
    user_cat_group['rank_neg'] = user_cat_group.groupby('masked_consumer_id')['amount_neg'].rank(ascending=False)

    # Pivot to user-level wide format
    pivot_cols = ['pct_amount_total', 'pct_amount_abs', 'pct_amount_pos', 'pct_amount_neg',
                  'rank_total', 'rank_abs', 'rank_pos', 'rank_neg']
    df_user_cat_stats = user_cat_group.pivot(index='masked_consumer_id', columns='category', values=pivot_cols)
    df_user_cat_stats.columns = ['{}_{}'.format(stat, cat) for stat, cat in df_user_cat_stats.columns]
    df_user_cat_stats = df_user_cat_stats.reset_index()

    # Attach target column
    target_map = df_txn[['masked_consumer_id', 'FPF_TARGET']].drop_duplicates()
    df_user_cat_stats = df_user_cat_stats.merge(target_map, on='masked_consumer_id', how='left')

    return df_user_cat_stats

rank_cats = compute_category_spending_distribution(df_train)

# Preview the filtered data
end_df = build_df.merge(rank_cats.drop(columns = ['FPF_TARGET']), on = 'masked_consumer_id', how = 'left')

In [16]:
cols_names = ['amount_mean_ovr_pos',
 'pct_amount_abs_26.0',
 'pct_amount_neg_27.0',
 'weekly_avg_spend_15',
 'txn_active_days_group_1',
 'pct_amount_abs_13.0',
 'log_amount_mean_group_3_neg',
 'rank_abs_26.0',
 'pct_amount_neg_26.0',
 'amount_skew_1_neg',
 'amount_max_6',
 'txn_active_day_pct_group_1',
 'log_amount_mean_group_10_pos',
 'monthly_spend_slope_group_5',
 'amount_gini_26',
 'pct_amount_abs_30.0',
 'pct_amount_total_3.0',
 'txn_count_2',
 'low_dollar_txn_pct_group_9',
 'rank_abs_17.0',
 'pct_amount_abs_27.0',
 'amount_min_group_9',
 'txn_gap_iqr_19',
 'weekday_spend_std_group_10_neg',
 'amount_cv_26',
 'high_to_low_spend_ratio_27',
 'txn_burstiness_group_3_neg',
 'rank_total_23.0',
 'amount_min_2',
 'amount_skew_6',
 'amount_std_11_neg',
 'amount_median_23',
 'log_amount_mean_group_10_neg',
 'high_to_low_spend_ratio_6',
 'amount_min_group_6',
 'amount_median_15',
 'txn_gap_iqr_20',
 'amount_stdovr',
 'txn_gap_iqr_21',
 'high_to_low_spend_ratio_group_6',
 'monthly_avg_spend_14',
 'weekend_spend_pctovr',
 'txn_gap_iqr_group_10',
 'monthly_txn_freq_23',
 'amount_skew_group_1',
 'monthly_avg_spend_15',
 'pct_amount_total_10.0',
 'pct_amount_abs_24.0',
 'weekend_spend_pct_group_7',
 'pct_amount_total_25.0',
 'monthly_avg_spend_20',
 'weekly_spend_slope_group_3',
 'weekly_spend_slope_group_7',
 'amount_gini_group_2',
 'weekday_spend_std_group_2',
 'amount_mean_7',
 'amount_min_11_pos',
 'rank_pos_13.0',
 'amount_skew_18',
 'weekly_spend_slope_3',
 'low_dollar_txn_pct_group_10',
 'txn_gap_iqr_group_7',
 'pct_amount_abs_35.0',
 'week_series_category_21_trend',
 'amount_cv_group_2',
 'txn_burstiness_1_neg',
 'weekday_spend_std_group_5',
 'pct_amount_pos_2.0',
 'monthly_avg_spend_17',
 'amount_skew_20',
 'amount_kurtosis_13',
 'rank_total_2.0',
 'high_to_low_spend_ratio_group_3_neg',
 'amount_mean_4',
 'txn_count_13',
 'amount_min_12_neg',
 'log_amount_mean_group_1',
 'weekday_spend_std_group_3_neg',
 'amount_cv_13',
 'rank_pos_2.0',
 'high_to_low_spend_ratio_25',
 'amount_median_group_1',
 'cat_concentrationovr',
 'weekly_avg_spend_28',
 'amount_std_26',
 'txn_active_day_pct_21',
 'weekly_txn_freq_group_8',
 'amount_max_23',
 'rank_abs_20.0',
 'weekly_avg_spend_ovr_pos',
 'weekly_avg_spend_12_neg',
 'amount_skew_group_3',
 'txn_burstiness_13_neg',
 'monthly_spend_slope_group_9',
 'txn_burstiness_20',
 'log_amount_mean_18',
 'monthly_spend_slope_0_pos',
 'txn_burstiness_group_1',
 'weekly_txn_freq_7',
 'monthly_spend_slope_18']
X = end_df[cols_names]
y = end_df['FPF_TARGET']

def custom_cv_auc(X, y, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=2)
    aucs = []

    for train_idx, test_idx in skf.split(X, y):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        model = create_xgb_model(y_train)
        model.fit(
                X_train, y_train,
                eval_set=[(X_test, y_test)],
                verbose=False
            )

        y_pred_probs = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_pred_probs)
        aucs.append(auc)

    return np.array(aucs)

def create_xgb_model(y_train):
    pos = sum(y_train == 1)
    neg = sum(y_train == 0)
    scale_pos_weight = neg / pos
    return xgb.XGBClassifier(
        use_label_encoder=False,
        eval_metric='auc',
        scale_pos_weight=scale_pos_weight,
        n_estimators=631,
        max_depth=30,
        learning_rate=0.0011634589962664767,
        min_child_weight=1,
        gamma=8.327385155929777,
        subsample=0.698047062927116,
        colsample_bytree=0.7255974804615869,
        reg_alpha=0.2442463600286613,
        reg_lambda=0.3431217197507496,
        random_state=0,
        n_jobs=-1
    )

aucs = custom_cv_auc(X, y)
aucs.mean()

0.7603121823169723